[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/41_capstone_projects/01_capstone_guide.ipynb)

# Bài 01 — Hướng dẫn dự án Capstone

**🎯 Mục tiêu**
- Hiểu capstone khác một bài Kaggle thế nào: bắt đầu từ **quyết định** cần cải thiện, kết thúc bằng **hệ thống** có thể vận hành và một báo cáo trung thực.
- Chọn đề tài phù hợp (10 gợi ý dùng dữ liệu của khoá học), lập kế hoạch 6 tuần với sản phẩm cụ thể mỗi tuần.
- Viết **tài liệu thiết kế** (design doc) trước khi code; dựng khung dự án chuẩn bằng code.
- Tự chấm theo **rubric** 8 tiêu chí và **kiểm tra tự động** các "cờ đỏ" thường gặp trước khi nộp.

Bài 02 là một capstone mẫu hoàn chỉnh (chấm điểm tín dụng từ dữ liệu thô) đi qua đúng các bước ở đây.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/41_capstone_projects

In [ ]:
import json
import re
from pathlib import Path

import pandas as pd

ARTIFACTS = Path("artifacts")                                           # thư mục đã nằm trong .gitignore

## 1. Capstone ≠ Kaggle

| | Bài thi Kaggle | Dự án ML thật (capstone) |
|---|---|---|
| Bài toán | cho sẵn, metric cho sẵn | **tự định nghĩa**: quyết định nào thay đổi, ai dùng, metric kinh doanh nào |
| Dữ liệu | sạch, chia sẵn train/test | **thô**: trùng, thiếu, sai đơn vị, nhãn trễ — tự làm sạch, tự chia đúng thời gian |
| Mục tiêu | +0.001 trên leaderboard | **giá trị** (lợi nhuận, thời gian, rủi ro) so với cách làm hiện tại |
| Đánh giá | một con số trên test ẩn | metric kèm độ bất định, phân tích lỗi, theo nhóm, calibration, chi phí |
| Mô hình | ensemble 50 mô hình | mô hình **đủ tốt, hiểu được, phục vụ được** |
| Kết thúc | nộp file dự đoán | gói mô hình + API + kiểm thử + kế hoạch giám sát + báo cáo |
| Thất bại | điểm thấp | rò rỉ dữ liệu, lệch train/serving, tối ưu sai metric, mô hình gây hại cho một nhóm |

Một capstone tốt có thể có mô hình **đơn giản** — nhưng mọi quyết định đều có lý do, được đo đạc, và người khác chạy lại được.

## 2. Chọn đề tài

Mười gợi ý dùng dữ liệu có sẵn trong khoá học (sinh offline, có "quy luật thật" để tự kiểm chứng). Bạn cũng có thể dùng dữ liệu thật của mình — khi đó dành thêm thời gian cho pháp lý và quyền riêng tư.

In [ ]:
ideas = pd.DataFrame([
    ("Chấm điểm tín dụng từ dữ liệu thô", "data/sample/credit_applications_raw.csv", "phân loại + quyết định theo lợi nhuận", "04, 07, 14, 19, 37, 39", "dữ liệu bẩn, nhãn trễ 12 tháng, cú sốc 2024"),
    ("Định giá nhà từ tin rao", "make_house_prices(dirty=True)", "hồi quy", "04, 06, 12, 17, 36", "dữ liệu bẩn, ngoại lai, giải thích giá"),
    ("Cảnh báo khách hàng rời bỏ", "make_customer_churn + nhật ký hoạt động (chương 39)", "phân loại theo thời gian", "07, 14, 38, 39", "đặc trưng đúng thời điểm, nhắm mục tiêu theo uplift"),
    ("Phát hiện gian lận thẻ", "make_fraud_transactions", "mất cân bằng nặng", "19, 22, 14, 40", "ngưỡng theo chi phí, độ trễ thời gian thực"),
    ("Dự báo doanh số theo ngày", "data/sample/sales_daily.csv", "chuỗi thời gian", "25, 15", "backtest, khoảng dự báo, mùa vụ"),
    ("Gợi ý sản phẩm", "make_ratings (ratings.csv, items.csv)", "gợi ý", "26, 21", "cold start, thiên lệch chọn lọc, metric xếp hạng"),
    ("Phân loại tin tức tiếng Việt", "make_news_vi / reviews_vi.csv", "NLP", "32, 31", "từ đồng nghĩa, nhiễu nhãn, so sánh TF-IDF vs embedding"),
    ("Trợ lý hỏi đáp FAQ", "faq_vi.csv + faq_queries_vi.csv", "truy hồi ngữ nghĩa", "32, 31", "đánh giá truy hồi, câu hỏi ngoài phạm vi"),
    ("Nhận dạng hình khối", "make_shapes / make_shape_scenes", "thị giác máy tính", "29, 33, 34", "dịch chuyển miền màu, phát hiện vật thể"),
    ("Tối ưu chiến dịch khuyến mãi", "make_causal_marketing", "nhân quả / uplift", "38, 40", "dữ liệu quan sát, đánh giá bằng A/B"),
], columns=["đề tài", "dữ liệu", "loại bài toán", "chương chính", "thử thách thật"])
print(ideas.to_string(index=False))

Tiêu chí chọn: (1) có một **quyết định** rõ ràng được cải thiện; (2) có **baseline** không dùng ML để so sánh; (3) có nhãn (hoặc cách đánh giá) đáng tin; (4) vừa sức trong 6 tuần — thu hẹp phạm vi trước, mở rộng sau.

## 3. Lộ trình 6 tuần

| Tuần | Trọng tâm | Sản phẩm phải có cuối tuần |
|---|---|---|
| 1 | Bài toán, metric, dữ liệu | Design doc v1 (mục 4); tải/đọc dữ liệu; **baseline nghiệp vụ** + baseline đơn giản nhất |
| 2 | Dữ liệu | Báo cáo kiểm tra dữ liệu; hàm làm sạch **có test**; chia dữ liệu đúng (thời gian/nhóm); EDA trả lời câu hỏi của design doc |
| 3 | Mô hình | Pipeline đặc trưng; 2–3 họ mô hình; validation/CV; experiment tracking (chương 39) |
| 4 | Đánh giá sâu | Test (chạm một lần); khoảng tin cậy; phân tích lỗi; theo nhóm; calibration; quyết định theo chi phí; giải thích |
| 5 | Kỹ thuật | Gói mô hình + metadata + model card; API + kiểm thử parity; Docker/CI (chương 40) |
| 6 | Truyền đạt | Kế hoạch giám sát; báo cáo; demo 10 phút; mục "hạn chế & bước tiếp theo" |

Quy tắc: **có kết quả đầu–cuối sớm** (tuần 1 đã có một baseline chạy hết pipeline), rồi cải thiện từng phần. Một hệ thống đơn giản chạy được giá trị hơn một mô hình hoàn hảo chưa tích hợp.

In [ ]:
import matplotlib.pyplot as plt

from ml_course.visualization import set_style

set_style()
tasks = [("Bài toán, metric, design doc v1", 1, 1, "C0"), ("Baseline chạy đầu–cuối", 1, 1, "C3"),
         ("Kiểm tra & làm sạch dữ liệu, EDA", 2, 1, "C0"), ("Pipeline đặc trưng, 2–3 họ mô hình, CV", 3, 1, "C1"),
         ("Experiment tracking", 3, 4, "C1"), ("Đánh giá sâu: test, KTC, lỗi, calibration", 4, 1, "C2"),
         ("Gói mô hình, API, Docker/CI", 5, 1, "C4"), ("Giám sát, báo cáo, demo", 6, 1, "C5"), ("Cập nhật design doc", 1, 6, "0.6")]
fig, ax = plt.subplots(figsize=(11, 4))
for i, (name, start, length, color) in enumerate(tasks):
    ax.broken_barh([(start - 0.45, length - 0.1)], (len(tasks) - 1 - i - 0.35, 0.7), color=color, alpha=0.8)
ax.set_yticks(range(len(tasks)), [t[0] for t in tasks][::-1])
ax.set_xticks(range(1, 7), [f"tuần {w}" for w in range(1, 7)])
ax.set_xlim(0.4, 6.6)
ax.grid(axis="x", alpha=0.4)
ax.set_title("Lộ trình capstone 6 tuần: có kết quả đầu–cuối ngay tuần 1, rồi cải thiện từng phần")
plt.tight_layout()
plt.show()

## 4. Tài liệu thiết kế

Viết **trước** khi code (1–2 trang), cập nhật khi học được điều mới. Nó buộc ta trả lời những câu hỏi mà code không trả lời (chương 39, bài 01):

In [ ]:
DESIGN_DOC = """# Design doc — {name}

## 1. Bối cảnh và quyết định
- Quyết định nào sẽ thay đổi nhờ mô hình? Ai ra quyết định, bao lâu một lần?
- Hiện nay quyết định được đưa ra thế nào (baseline nghiệp vụ)? Chi phí của sai lầm mỗi loại?

## 2. Metric
- Metric kinh doanh (vd lợi nhuận/khách, thời gian tiết kiệm):
- Metric mô hình (vd ROC-AUC, MAE) và vì sao nó liên quan tới metric kinh doanh:
- Metric bảo vệ (guardrail): độ trễ, công bằng giữa các nhóm, tỉ lệ duyệt...
- Ngưỡng "đủ tốt để triển khai":

## 3. Dữ liệu và nhãn
- Nguồn, khoảng thời gian, số dòng; định nghĩa nhãn chính xác; nhãn có sau bao lâu?
- Đặc trưng nào **có sẵn tại thời điểm dự đoán**? Cột nào là thông tin sau quyết định (cấm dùng)?
- Chia train/validation/test thế nào (thời gian? nhóm?) và vì sao.

## 4. Cách tiếp cận
- Baseline: ... · Mô hình ứng viên: ... · Cách chọn: ...

## 5. Rủi ro
- Rò rỉ, trôi dữ liệu, vòng phản hồi, nhóm bị ảnh hưởng bất lợi, pháp lý/quyền riêng tư.

## 6. Triển khai và giám sát
- Batch hay online? Độ trễ? Giám sát gì, nhãn proxy nào, khi nào huấn luyện lại, ai chịu trách nhiệm?
"""
print(DESIGN_DOC.format(name="<tên dự án>")[:600], "...")

## 5. Dựng khung dự án

Cấu trúc thống nhất giúp người chấm (và chính bạn sau 3 tháng) tìm thấy mọi thứ. Hàm dưới tạo khung tối thiểu — notebook cho khám phá, **package** cho code dùng lại, **tests**, tài liệu:

In [ ]:
def scaffold_project(root: Path, name: str, package: str) -> list[str]:
    files = {
        "README.md": f"# {name}\n\n> Một câu: quyết định nào được cải thiện, kết quả chính (kèm khoảng tin cậy), cách chạy lại.\n\n"
                     "## Chạy lại\n```bash\npip install -r requirements.txt\npython -m {pkg}.train --data data/raw/... --out models/\npytest -q\n```\n".replace("{pkg}", package),
        "docs/design_doc.md": DESIGN_DOC.format(name=name),
        "docs/model_card.md": f"# Model card — {name}\n\n## Mục đích sử dụng\n## Dữ liệu\n## Hiệu năng (tổng và theo nhóm)\n## Hạn chế\n",
        "docs/report.md": "# Báo cáo\n\n1. Bài toán\n2. Dữ liệu\n3. Phương pháp\n4. Kết quả\n5. Phân tích lỗi\n6. Triển khai và giám sát\n7. Hạn chế và bước tiếp theo\n",
        f"src/{package}/__init__.py": '__version__ = "0.1.0"\n',
        f"src/{package}/data.py": '"""Đọc, kiểm tra, làm sạch, chia dữ liệu."""\n',
        f"src/{package}/features.py": '"""Đặc trưng — dùng chung cho huấn luyện và phục vụ."""\n',
        f"src/{package}/train.py": '"""python -m {pkg}.train — huấn luyện, cổng chất lượng, đóng gói."""\n'.replace("{pkg}", package),
        f"src/{package}/api.py": '"""Dịch vụ FastAPI."""\n',
        "tests/test_data.py": "def test_placeholder():\n    assert True\n",
        "notebooks/01_eda.md": "Notebook khám phá đặt ở đây (không chứa logic dùng lại).\n",
        "data/README.md": "Dữ liệu KHÔNG commit vào git — mô tả nguồn, phiên bản (hash), cách tải.\n",
        "requirements.txt": "numpy\npandas\nscikit-learn\n",
        ".gitignore": "data/raw/\ndata/processed/\nmodels/\n__pycache__/\n",
    }
    for rel, content in files.items():
        path = root / rel
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(content, encoding="utf-8")
    return sorted(files)


project_root = ARTIFACTS / "my_capstone"
for f in scaffold_project(project_root, "Chấm điểm tín dụng", "credit"):
    print("  ", f)

## 6. Rubric chấm điểm

Tám tiêu chí, mỗi tiêu chí 4 mức (1 = thiếu · 2 = cơ bản · 3 = tốt · 4 = xuất sắc). **Đạt** khi điểm quy đổi ≥ 70/100 **và** không tiêu chí nào ở mức 1 — một lỗ hổng nghiêm trọng (vd rò rỉ dữ liệu) không bù được bằng điểm cao ở chỗ khác.

In [ ]:
RUBRIC = {
    "Bài toán & metric": (10, "quyết định, baseline nghiệp vụ, metric kinh doanh ↔ mô hình, ngưỡng triển khai rõ"),
    "Dữ liệu & làm sạch": (15, "kiểm tra chất lượng có hệ thống, làm sạch có test, nhãn và thời điểm đúng, không rò rỉ"),
    "Baseline & mô hình": (15, "baseline đơn giản, ≥ 2 họ mô hình, pipeline, lựa chọn có lý do"),
    "Đánh giá nghiêm ngặt": (20, "chia đúng (thời gian/nhóm), test chạm một lần, độ bất định, calibration, chi phí, theo nhóm"),
    "Phân tích lỗi & giải thích": (10, "lỗi tập trung ở đâu, giải thích toàn cục/cục bộ, kiểm tra công bằng"),
    "Tái lập & kỹ thuật": (10, "chạy lại được bằng 1–2 lệnh, seed, phiên bản, package + tests"),
    "Triển khai & giám sát": (10, "gói mô hình + API + kiểm thử parity, kế hoạch giám sát với nhãn trễ, huấn luyện lại"),
    "Truyền đạt": (10, "báo cáo trung thực (kể cả điều không hiệu quả), hạn chế, model card, demo"),
}


def grade(levels: dict) -> dict:
    """levels: {tiêu chí: 1..4} → điểm quy đổi /100 và kết luận."""
    missing = set(RUBRIC) - set(levels)
    if missing:
        raise ValueError(f"thiếu tiêu chí: {sorted(missing)}")
    score = sum(RUBRIC[c][0] * (levels[c] - 1) / 3 for c in RUBRIC)
    weak = [c for c, lv in levels.items() if lv == 1]
    return {"điểm": round(score, 1), "đạt": score >= 70 and not weak, "tiêu chí mức 1": weak}


print(pd.DataFrame({"trọng số": {c: w for c, (w, _) in RUBRIC.items()}, "mức 4 nghĩa là": {c: d for c, (_, d) in RUBRIC.items()}}).to_string())
strong_model_weak_data = {c: 4 for c in RUBRIC} | {"Dữ liệu & làm sạch": 1}
balanced = {c: 3 for c in RUBRIC} | {"Đánh giá nghiêm ngặt": 4}
print("\nmô hình xuất sắc nhưng rò rỉ dữ liệu:", grade(strong_model_weak_data))
print("cân bằng, mô hình đơn giản            :", grade(balanced))

## 7. Kiểm tra "cờ đỏ" tự động

Một số lỗi kiểm tra được bằng máy trước khi nộp. Hàm dưới quét thư mục dự án; chạy trên khung vừa tạo để thấy còn thiếu gì:

In [ ]:
def audit_project(root: Path, max_file_mb: float = 20) -> pd.DataFrame:
    checks = []

    def add(name, ok, detail=""):
        checks.append({"kiểm tra": name, "kết quả": "✅" if ok else "❌", "chi tiết": detail})

    add("README có mục chạy lại", (root / "README.md").exists() and "Chạy lại" in (root / "README.md").read_text(encoding="utf-8"))
    design = root / "docs" / "design_doc.md"
    unfilled = design.exists() and design.read_text(encoding="utf-8").count("...") > 0
    add("design doc đã điền", design.exists() and not unfilled, "còn chỗ '...' chưa điền" if unfilled else "")
    tests = list((root / "tests").glob("test_*.py")) if (root / "tests").exists() else []
    trivial = [t.name for t in tests if "assert True" in t.read_text(encoding="utf-8")]
    add("có test thật", bool(tests) and not trivial, f"test giả: {trivial}" if trivial else f"{len(tests)} file")
    req = (root / "requirements.txt").read_text(encoding="utf-8").split() if (root / "requirements.txt").exists() else []
    unpinned = [r for r in req if "==" not in r]
    add("phụ thuộc ghim phiên bản", bool(req) and not unpinned, f"chưa ghim: {unpinned}" if unpinned else "")
    ignore = (root / ".gitignore").read_text(encoding="utf-8") if (root / ".gitignore").exists() else ""
    add(".gitignore loại dữ liệu & mô hình", "data/raw" in ignore and "models" in ignore)
    big = [str(p.relative_to(root)) for p in root.rglob("*") if p.is_file() and p.stat().st_size > max_file_mb * 2**20]
    add(f"không có file > {max_file_mb} MB", not big, ", ".join(big))
    secrets = [str(p.relative_to(root)) for p in root.rglob("*.py")
               if re.search(r"(api_key|password|secret)\s*=\s*['\"][^'\"]+['\"]", p.read_text(encoding="utf-8"), re.I)]
    add("không lộ khoá bí mật trong code", not secrets, ", ".join(secrets))
    card = root / "docs" / "model_card.md"
    add("model card có hiệu năng theo nhóm", card.exists() and "nhóm" in card.read_text(encoding="utf-8"))
    return pd.DataFrame(checks)


print(audit_project(project_root).to_string(index=False))

Khung mới tạo **chưa đạt** — đúng như mong đợi: design doc còn chỗ trống, test là giả, phụ thuộc chưa ghim phiên bản. Chạy lại kiểm tra này trước mỗi buổi review.

Những cờ đỏ máy **không** bắt được — tự hỏi:
- Có đặc trưng nào chỉ biết **sau** thời điểm quyết định không? (rò rỉ — chương 05, 39)
- Tập test có bị dùng để chọn mô hình/ngưỡng không? (lời nguyền người thắng — chương 39)
- Baseline đơn giản đạt bao nhiêu? Cải thiện có lớn hơn độ bất định không?
- Mô hình tệ nhất với nhóm nào? Ai chịu thiệt khi mô hình sai?
- Nếu phân phối dữ liệu đổi vào tháng tới, ta biết bằng cách nào và sau bao lâu?

## 8. Báo cáo và demo

**Báo cáo** (5–10 trang) theo cấu trúc `docs/report.md`; viết cho người ra quyết định đọc phần đầu, kỹ sư đọc phần sau:
1. **Tóm tắt** (½ trang): quyết định, kết quả chính so với baseline **kèm khoảng tin cậy**, khuyến nghị.
2. Bài toán & metric · 3. Dữ liệu (kể cả vấn đề chất lượng) · 4. Phương pháp · 5. Kết quả · 6. Phân tích lỗi & theo nhóm · 7. Triển khai & giám sát · 8. **Hạn chế** và bước tiếp theo.

**Demo 10 phút**: 1 phút bài toán → 2 phút dữ liệu và điều bất ngờ → 3 phút kết quả (so với baseline, theo nhóm) → 2 phút demo API/dashboard → 2 phút giám sát, hạn chế. Một biểu đồ tốt hơn mười bảng số.

Sai lầm thường gặp khi trình bày: chỉ báo cáo con số tốt nhất; không nói đã thử bao nhiêu cấu hình; AUC không kèm metric kinh doanh; giấu những gì không hiệu quả. Báo cáo **trung thực** về giới hạn là dấu hiệu của chuyên môn, không phải điểm yếu.

## 🧠 Kiểm tra nhanh

1. Kể năm khác biệt giữa capstone và một cuộc thi Kaggle.
2. Bốn tiêu chí chọn đề tài?
3. Vì sao phải có kết quả đầu–cuối ngay tuần 1?
4. Design doc trả lời những câu hỏi gì mà code không trả lời?
5. Vì sao rubric yêu cầu không tiêu chí nào ở mức 1?
6. Kể ba cờ đỏ máy kiểm tra được và ba cờ đỏ phải tự hỏi.

## 📌 Tóm tắt

- Capstone bắt đầu từ một quyết định và kết thúc bằng một hệ thống có thể vận hành, cùng báo cáo trung thực.
- Kế hoạch 6 tuần với sản phẩm cụ thể; baseline đầu–cuối sớm; design doc trước code.
- Khung dự án chuẩn (package, tests, docs); rubric 8 tiêu chí; kiểm tra cờ đỏ tự động và tự vấn.

## 📚 Tài liệu tham khảo

- Huyen (2022) — *Designing Machine Learning Systems* (chương 1–2: tổng quan và khung thiết kế)
- Google — *Rules of Machine Learning* (Zinkevich) — đặc biệt các quy tắc 1–4 về khởi đầu đơn giản
- Breck et al. (2017) — *The ML Test Score*
- Mitchell et al. (2019) — *Model Cards for Model Reporting*
- Cookiecutter Data Science — cấu trúc dự án khoa học dữ liệu